# ArXiv RAG — Stage 2: Embedding Pipeline

Reads the 50k papers already in Neon Postgres (from Stage 1),
chunks each abstract, embeds with `nomic-embed-text-v2-moe` on GPU,
and bulk-inserts into the `chunks` table.

**Before running:**
1. Go to **Add-ons → Secrets** in this notebook
2. Add a secret named `DATABASE_URL` with your Neon connection string:
   `postgresql://neondb_owner:<password>@<host>/neondb?sslmode=require`
3. Enable **GPU accelerator** (T4 recommended) under Settings → Accelerator
4. Run All

In [ ]:
# Install dependencies
!pip install -q 'psycopg[binary]' psycopg_pool 'pgvector>=0.3.5' sentence-transformers einops numpy

In [ ]:
import os

# Load DATABASE_URL from Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['DATABASE_URL'] = UserSecretsClient().get_secret('DATABASE_URL')
    print('DATABASE_URL loaded from Kaggle secrets')
except Exception as e:
    print(f'Could not load secret: {e}')
    print('Set DATABASE_URL manually: os.environ["DATABASE_URL"] = "postgresql://..."')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import asyncio
import logging
import time
import re
from dataclasses import dataclass, field
from typing import AsyncIterator

import numpy as np
import psycopg
from psycopg_pool import AsyncConnectionPool
from pgvector.psycopg import register_vector_async
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Config ──────────────────────────────────────────────────────────────────
EMBED_MODEL   = 'nomic-ai/nomic-embed-text-v2-moe'
EMBED_DIM     = 768
CHUNK_SIZE    = 400   # tokens (approx words)
CHUNK_OVERLAP = 50
BATCH_SIZE    = 1000  # papers per flush
EMBED_BATCH   = 512   # texts per model.encode() call
LIMIT         = 50_000

print('Config loaded')

In [ ]:
# ── Chunking ─────────────────────────────────────────────────────────────────
@dataclass
class Chunk:
    doc_id: str
    section_title: str
    chunk_index: int
    content: str
    token_count: int

def _word_chunks(text: str, size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += size - overlap
    return chunks

def chunk_text(text: str, doc_id: str, section_title: str = 'abstract') -> list[Chunk]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    raw = _word_chunks(text, CHUNK_SIZE, CHUNK_OVERLAP)
    return [
        Chunk(doc_id=doc_id, section_title=section_title,
              chunk_index=i, content=c, token_count=len(c.split()))
        for i, c in enumerate(raw)
    ]

print('Chunking helpers ready')

In [ ]:
# ── Embedding model ───────────────────────────────────────────────────────────
print(f'Loading {EMBED_MODEL} ...')
_model = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
print(f'Model loaded (dim={EMBED_DIM})')

def embed_chunks(chunks: list[Chunk]) -> list[tuple[Chunk, list[float]]]:
    if not chunks:
        return []
    texts = [f'search_document: {c.content}' for c in chunks]
    embeddings = _model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=EMBED_BATCH,
        device=device,
    )
    return list(zip(chunks, embeddings.tolist()))

# Quick smoke test
test = embed_chunks([Chunk('test', 'abstract', 0, 'attention is all you need', 5)])
assert len(test[0][1]) == EMBED_DIM
print(f'Smoke test passed — embedding dim={len(test[0][1])}')

In [ ]:
# ── Database ──────────────────────────────────────────────────────────────────
_pool = None

async def init_pool():
    global _pool
    url = os.environ['DATABASE_URL']
    _pool = AsyncConnectionPool(conninfo=url, min_size=2, max_size=10, open=False)
    await _pool.open()
    logger.info('DB pool initialized')

async def close_pool():
    global _pool
    if _pool:
        await _pool.close()

from contextlib import asynccontextmanager

@asynccontextmanager
async def get_conn():
    async with _pool.connection() as conn:
        yield conn

async def insert_chunks_batch(conn, chunks: list[dict]):
    await register_vector_async(conn)
    sql = """
        INSERT INTO chunks (paper_id, section_title, chunk_index, content, token_count, embedding)
        VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = [
        (c['paper_id'], c['section_title'], c['chunk_index'],
         c['content'], c['token_count'], c['embedding'])
        for c in chunks
    ]
    async with conn.cursor() as cur:
        await cur.executemany(sql, params)

print('DB helpers ready')

In [ ]:
# ── Main pipeline ─────────────────────────────────────────────────────────────
async def run_pipeline(limit: int = LIMIT, batch_size: int = BATCH_SIZE):
    await init_pool()

    # Fetch IDs of papers that already have chunks (to skip them)
    async with get_conn() as conn:
        async with conn.cursor() as cur:
            await cur.execute('SELECT DISTINCT paper_id FROM chunks')
            already_done = {row[0] for row in await cur.fetchall()}
    logger.info('Papers already embedded: %d — will skip these', len(already_done))

    total_docs = 0
    total_chunks = 0
    skipped = 0
    start = time.time()
    buffer: list[tuple[int, list]] = []  # (paper_id, chunks)

    async def flush(buf):
        if not buf:
            return 0
        paper_ids_flat, chunks_flat = [], []
        for pid, cks in buf:
            for c in cks:
                paper_ids_flat.append(pid)
                chunks_flat.append(c)

        pairs = embed_chunks(chunks_flat)

        rows = [
            {
                'paper_id': paper_ids_flat[i],
                'section_title': chunk.section_title,
                'chunk_index': chunk.chunk_index,
                'content': chunk.content,
                'token_count': chunk.token_count,
                'embedding': emb,
            }
            for i, (chunk, emb) in enumerate(pairs)
        ]

        async with get_conn() as conn:
            await insert_chunks_batch(conn, rows)
            await conn.commit()

        return len(rows)

    async with get_conn() as conn:
        async with conn.cursor() as cur:
            await cur.execute(
                """
                SELECT id, arxiv_id, abstract
                FROM papers
                WHERE abstract IS NOT NULL AND abstract != ''
                ORDER BY published_at DESC
                LIMIT %s
                """,
                (limit,),
            )
            rows_fetched = await cur.fetchall()

    logger.info('Fetched %d papers from DB', len(rows_fetched))

    for paper_id, arxiv_id, abstract in rows_fetched:
        if paper_id in already_done:
            skipped += 1
            continue

        chunks = chunk_text(abstract, doc_id=arxiv_id)
        if not chunks:
            continue

        buffer.append((paper_id, chunks))
        total_docs += 1

        if len(buffer) >= batch_size:
            n = await flush(buffer)
            total_chunks += n
            buffer = []
            elapsed = time.time() - start
            rate = total_docs / elapsed * 60
            logger.info(
                'Processed %d docs | %d chunks | %.0f docs/min | %.1fs elapsed | skipped=%d',
                total_docs, total_chunks, rate, elapsed, skipped,
            )

    # Flush remainder
    n = await flush(buffer)
    total_chunks += n

    await close_pool()

    elapsed = time.time() - start
    stats = {
        'total_docs': total_docs,
        'total_chunks': total_chunks,
        'skipped_already_done': skipped,
        'elapsed_minutes': round(elapsed / 60, 1),
    }
    logger.info('Pipeline complete: %s', stats)
    return stats

print('Pipeline function defined — ready to run')

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# This cell does all the work. Expected time on T4 GPU: ~30-40 minutes for 50k papers.
stats = await run_pipeline(limit=50_000, batch_size=1000)
print('\n✓ Done:', stats)